## **Stratified Sampling untuk Anotasi**

In [59]:
from pathlib import Path
import math
import pandas as pd

SEED = 42

TARGET_TOTAL = 15000
N_OVERLAP = 500
MIN_PER_ASPEK = 80

ANNDIR = Path("data/annotation")
ANNDIR.mkdir(parents=True, exist_ok=True)

In [70]:
df = pd.read_csv(
    "data/processed/sentences_with_aspect.csv",
    keep_default_na=False
)

In [71]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 63627 entries, 0 to 63626
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_artikel     63627 non-null  int64
 1   id_kalimat     63627 non-null  str  
 2   bank           63627 non-null  str  
 3   timestamp      63627 non-null  str  
 4   kalimat        63627 non-null  str  
 5   n_kata         63627 non-null  int64
 6   topic          63627 non-null  int64
 7   nama_aspek     63627 non-null  str  
 8   kategori_fiqa  63627 non-null  str  
dtypes: int64(3), str(6)
memory usage: 15.6 MB


In [62]:
df = pd.read_csv(
    "data/processed/sentences_with_aspect.csv",
    dtype={"id_artikel": str, "id_kalimat": str}
)

# Kandidat anotasi: semua kalimat kecuali aspek Lainnya
kandidat = df[df["nama_aspek"] != "Lainnya"].copy()

assert kandidat["id_kalimat"].is_unique, "id_kalimat harus unik!"
assert kandidat["id_kalimat"].notna().all(), "Ada id_kalimat kosong!"

print(f"Total data       : {len(df):,}")
print(f"Kandidat anotasi : {len(kandidat):,}")
print(f"Jumlah aspek     : {kandidat['nama_aspek'].nunique()}")

Total data       : 63,627
Kandidat anotasi : 42,050
Jumlah aspek     : 15


In [63]:
if len(kandidat) < TARGET_TOTAL:
    raise ValueError(
        f"Kandidat hanya {len(kandidat):,}, "
        f"sedangkan target {TARGET_TOTAL:,}."
    )

if N_OVERLAP >= TARGET_TOTAL:
    raise ValueError("N_OVERLAP harus lebih kecil dari TARGET_TOTAL.")

jumlah_aspek = kandidat["nama_aspek"].nunique()

if TARGET_TOTAL < MIN_PER_ASPEK * jumlah_aspek:
    raise ValueError(
        "TARGET_TOTAL terlalu kecil untuk memenuhi MIN_PER_ASPEK."
    )

print(f"Target kalimat unik : {TARGET_TOTAL:,}")
print(f"Target overlap     : {N_OVERLAP:,}")
print(f"Target non-overlap : {TARGET_TOTAL - N_OVERLAP:,}")

Target kalimat unik : 15,000
Target overlap     : 500
Target non-overlap : 14,500


In [64]:
ukuran = kandidat["nama_aspek"].value_counts().sort_index()

# Alokasi awal: minimum per aspek, dibatasi ketersediaan
alok = {
    aspek: min(MIN_PER_ASPEK, int(jumlah))
    for aspek, jumlah in ukuran.items()
}

sisa_target = TARGET_TOTAL - sum(alok.values())

# Distribusikan sisa target secara proporsional
while sisa_target > 0:

    aktif = [
        aspek for aspek in ukuran.index
        if alok[aspek] < ukuran[aspek]
    ]

    if not aktif:
        raise ValueError("Kandidat tidak cukup untuk memenuhi target.")

    bobot = ukuran.loc[aktif]
    proporsi = sisa_target * bobot / bobot.sum()

    tambahan_dasar = {
        aspek: min(
            int(math.floor(proporsi[aspek])),
            int(ukuran[aspek] - alok[aspek])
        )
        for aspek in aktif
    }

    jumlah_tambahan = sum(tambahan_dasar.values())

    for aspek, tambahan in tambahan_dasar.items():
        alok[aspek] += tambahan

    sisa_target -= jumlah_tambahan

    if sisa_target == 0:
        break

    # Distribusikan sisa pembulatan berdasarkan fractional remainder
    ranking = sorted(
        aktif,
        key=lambda aspek: (
            -(proporsi[aspek] - math.floor(proporsi[aspek])),
            aspek
        )
    )

    for aspek in ranking:
        if sisa_target == 0:
            break

        if alok[aspek] < ukuran[aspek]:
            alok[aspek] += 1
            sisa_target -= 1

print("Alokasi per aspek:")

for aspek in ukuran.index:
    print(
        f"{aspek:35} "
        f"{alok[aspek]:>6,} / tersedia {ukuran[aspek]:,}"
    )

print("Total alokasi:", sum(alok.values()))

assert sum(alok.values()) == TARGET_TOTAL

Alokasi per aspek:
Aksi Investor Asing                  1,480 / tersedia 4,267
Bursa Global                         1,094 / tersedia 3,089
Dividen & Aksi Korporasi               899 / tersedia 2,496
Kebijakan Moneter & Nilai Tukar        898 / tersedia 2,494
Kinerja Keuangan & Laba                562 / tersedia 1,470
Kredit & Kualitas Aset               2,039 / tersedia 5,968
Layanan & Produk                       690 / tersedia 1,860
Makroekonomi & Faktor Global           602 / tersedia 1,590
Manajemen & Kepemimpinan               398 / tersedia 969
Pergerakan IHSG & Indeks             2,413 / tersedia 7,108
Pergerakan Saham Bank                1,015 / tersedia 2,848
Saham Spesifik Non-Bank                317 / tersedia 723
Sektor & Transaksi Pasar             1,175 / tersedia 3,335
Strategi & Ekspansi Bisnis           1,241 / tersedia 3,538
Valuasi & Kapitalisasi                 177 / tersedia 295
Total alokasi: 15000


Setiap aspek dijamin mendapat minimal 80 kalimat, lalu sisa kuota dibagikan sesuai besar-kecilnya jumlah data yang tersedia pada masing-masing aspek.

In [72]:
bagian_sampel = []

for aspek in ukuran.index:

    n = alok[aspek]

    pool = kandidat[
        kandidat["nama_aspek"] == aspek
    ]

    sampel_aspek = pool.sample(
        n=n,
        random_state=SEED
    )

    bagian_sampel.append(sampel_aspek)

sampel = (
    pd.concat(bagian_sampel, ignore_index=True)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

assert len(sampel) == TARGET_TOTAL
assert sampel["id_kalimat"].is_unique

print(f"Jumlah sampel: {len(sampel):,}")
print("Verifikasi ID unik: OK")

Jumlah sampel: 15,000
Verifikasi ID unik: OK


In [73]:
overlap = sampel.sample(
    n=N_OVERLAP,
    random_state=SEED
).copy()

id_overlap = set(overlap["id_kalimat"])

non_overlap = sampel[
    ~sampel["id_kalimat"].isin(id_overlap)
].copy()

non_overlap = non_overlap.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

half = len(non_overlap) // 2

porsi_A = non_overlap.iloc[:half].copy()
porsi_B = non_overlap.iloc[half:].copy()

print("Overlap      :", len(overlap))
print("Khusus A     :", len(porsi_A))
print("Khusus B     :", len(porsi_B))

Overlap      : 500
Khusus A     : 7250
Khusus B     : 7250


In [74]:
def sheet(frame, anotator):

    out = frame[
        [
            "id_kalimat",
            "id_artikel",
            "bank",
            "timestamp",
            "kalimat",
            "nama_aspek",
            "kategori_fiqa"
        ]
    ].copy()
    out["anotator"] = anotator
    out["sentimen"] = ""
    return out


anotator1 = pd.concat(
    [
        sheet(porsi_A, "A"),
        sheet(overlap, "A")
    ],
    ignore_index=True
)

anotator2 = pd.concat(
    [
        sheet(porsi_B, "B"),
        sheet(overlap, "B")
    ],
    ignore_index=True
)

print(f"Anotator A: {len(anotator1):,}")
print(f"Anotator B: {len(anotator2):,}")

Anotator A: 7,750
Anotator B: 7,750


### **Validasi Duplikat**

In [75]:
ids_A = set(anotator1["id_kalimat"])
ids_B = set(anotator2["id_kalimat"])

irisan_AB = ids_A & ids_B
union_AB = ids_A | ids_B

assert irisan_AB == id_overlap, (
    "Ada data yang sama antara A dan B di luar overlap!"
)

assert len(union_AB) == TARGET_TOTAL
assert len(irisan_AB) == N_OVERLAP

assert anotator1["id_kalimat"].is_unique
assert anotator2["id_kalimat"].is_unique

print("Validasi pembagian data: OK")
print("Total ID unik       :", len(union_AB))
print("ID overlap          :", len(irisan_AB))
print("ID khusus A         :", len(ids_A - ids_B))
print("ID khusus B         :", len(ids_B - ids_A))

Validasi pembagian data: OK
Total ID unik       : 15000
ID overlap          : 500
ID khusus A         : 7250
ID khusus B         : 7250


In [76]:
anotator1.to_csv(
    ANNDIR / "to_annotate_A.csv",
    index=False,
    encoding="utf-8-sig"
)

anotator2.to_csv(
    ANNDIR / "to_annotate_B.csv",
    index=False,
    encoding="utf-8-sig"
)

overlap[["id_kalimat"]].to_csv(
    ANNDIR / "overlap.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File anotasi berhasil disimpan:")
print(ANNDIR / "to_annotate_A.csv")
print(ANNDIR / "to_annotate_B.csv")
print(ANNDIR / "overlap.csv")

File anotasi berhasil disimpan:
data\annotation\to_annotate_A.csv
data\annotation\to_annotate_B.csv
data\annotation\overlap.csv


## **Analisa Hasil anotasi dengan cohen-kappa**

In [ ]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report

In [ ]:
ANNDIR = Path("data/annotation")

annotator_A = pd.read_csv(
    ANNDIR / "annotated_A.csv",
    dtype={"id_kalimat": str}
)

annotator_B = pd.read_csv(
    ANNDIR / "annotated_B.csv",
    dtype={"id_kalimat": str}
)

overlap = pd.read_csv(
    ANNDIR / "overlap.csv",
    dtype={"id_kalimat": str}
)

print("Anotator A:", len(annotator_A))
print("Anotator B:", len(annotator_B))
print("Overlap   :", len(overlap))

Anotator A: 7575
Anotator B: 7575
Overlap   : 500


In [ ]:
ids_A = set(annotator_A["id_kalimat"])
ids_B = set(annotator_B["id_kalimat"])
ids_overlap = set(overlap["id_kalimat"])

irisan_AB = ids_A & ids_B

In [ ]:
overlap_A = annotator_A[
    annotator_A["id_kalimat"].isin(ids_overlap)
][["id_kalimat", "sentimen"]].copy()

overlap_A = overlap_A.rename(
    columns={"sentimen": "sentimen_A"}
)

In [ ]:
overlap_B = annotator_B[
    annotator_B["id_kalimat"].isin(ids_overlap)
][["id_kalimat", "sentimen"]].copy()

overlap_B = overlap_B.rename(
    columns={"sentimen": "sentimen_B"}
)

In [ ]:
hasil_overlap = overlap_A.merge(
    overlap_B,
    on="id_kalimat",
    how="inner",
    validate="one_to_one"
)

hasil_overlap.head(5)

,id_kalimat,sentimen_A,sentimen_B
0,773804_18,0,0
1,766385_0,1,2
2,776455_0,0,0


In [ ]:
hasil_overlap_valid = hasil_overlap.dropna(
    subset=["sentimen_A", "sentimen_B"]
).copy()

kappa = cohen_kappa_score(
    hasil_overlap_valid["sentimen_A"],
    hasil_overlap_valid["sentimen_B"]
)

print(f"Cohen's Kappa: {kappa:.4f}")
print(f"Jumlah data valid: {len(hasil_overlap_valid)}")

Cohen's Kappa: 0.4000
Jumlah data valid: 3


### **Cek selisih sentimen**

In [ ]:
disagreement = hasil_overlap_valid[
    hasil_overlap_valid["sentimen_A"]
    != hasil_overlap_valid["sentimen_B"]
].copy()

print("Jumlah disagreement:", len(disagreement))

disagreement.head(10)

Jumlah disagreement: 1


,id_kalimat,sentimen_A,sentimen_B
1,766385_0,1,2


In [ ]:
def cek_fulltext(df, text, n=5, mode="head", random_state=42, anotasi_A="sentimen_A", anotasi_B="sentimen_B"):
    if text not in df.columns:
        raise ValueError(f"Kolom '{text}' tidak ditemukan.")

    if n <= 0:
        raise ValueError("n harus lebih besar dari 0.")

    if mode not in {"head", "tail", "sample"}:
        raise ValueError("mode harus head, tail, atau sample.")

    n = min(n, len(df))

    if mode == "head":
        hasil = df.head(n)

    elif mode == "tail":
        hasil = df.tail(n)

    else:
        hasil = df.sample(
            n=n,
            random_state=random_state
        )

    kolom = [
        col for col in [
            "id_artikel",
            "judul",
            "timestamp",
            text,
            anotasi_A,
            anotasi_B
        ]
        if col in df.columns
    ]

    for _, row in hasil[kolom].iterrows():
        print("=" * 100)

        if "id_artikel" in row:
            print(f"ID        : {row['id_artikel']}")

        if "judul" in row:
            print(f"Judul     : {row['judul']}")

        if "timestamp" in row:
            print(f"Timestamp : {row['timestamp']}")

        if anotasi_A in row:
            print(f"Anotasi A : {row[anotasi_A]}")

        if anotasi_B in row:
            print(f"Anotasi B : {row[anotasi_B]}")

        print(f"\nFulltext:\n{row[text]}\n")

In [ ]:
df_cek_overlap = disagreement.merge(
    df[["id_kalimat", "id_artikel", "kalimat"]],
    on="id_kalimat",
    how="left",
    validate="one_to_one"
)

In [ ]:
cek_fulltext(
    df=df_cek_overlap,
    text="kalimat",
    n=5,
    mode="sample"
)

ID        : 766385
Anotasi A : 1
Anotasi B : 2

Fulltext:
Indeks Harga Saham Gabungan (IHSG) ditutup melesat 1,01% ke level 6.686,44 pada perdagangan Selasa (8/9/2026), setelah pada perdagangan sebelumnya sempat terkoreksi 0,25%.



## **Gabungkan File Anotasi**

In [55]:
# Gabungkan anotasi A dan B
df_merged = pd.concat(
    [annotator_A, annotator_B],
    ignore_index=True
)

# Drop duplicate berdasarkan id_kalimat bekas cohens-kappa
df_merged = (
    df_merged
    .drop_duplicates(subset="id_kalimat", keep="first")
    .reset_index(drop=True)
)

In [56]:
df_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_kalimat     15000 non-null  str  
 1   id_artikel     15000 non-null  int64
 2   bank           14988 non-null  str  
 3   timestamp      15000 non-null  str  
 4   kalimat        15000 non-null  str  
 5   nama_aspek     15000 non-null  str  
 6   kategori_fiqa  15000 non-null  str  
 7   anotator       15000 non-null  str  
 8   sentimen       15000 non-null  int64
dtypes: int64(2), str(7)
memory usage: 3.8 MB
